# ARC-AGI-2 v3 — Backward Relational Edit-Schema Induction

This is the first prototype of the **difference-first** idea. Instead of enumerating transformation programs forward, it uses each known training output to derive exact edit explanations, then intersects/anti-unifies their abstract structure across demonstrations.

Current schema families include whole-grid geometry/color maps, object crop/isolation, recoloring, fill-holes, object move/copy, role-based colors, and relative destination rules. The important experimental question is **coverage**: can backward explanation produce nonzero demonstration-consistent schemas where v0/v1 forward DSL search had essentially none?

This notebook develops only on the deterministic held-out split of the 1,000 training tasks. It does **not** touch the official 120-task evaluation set.

**Compute:** CPU only. A GPU is not needed for this symbolic prototype.

In [ ]:
# First run: keep this at 50 for a fast held-out smoke test.
LIMIT = 50
SPLIT = 'heldout'

print('LIMIT =', LIMIT)
print('SPLIT =', SPLIT)
print('GPU needed: no — CPU is preferred for this version.')

In [ ]:
from pathlib import Path
import importlib.util, requests, sys

WORK = Path('/kaggle/working')
module_path = WORK / 'edit_schema_induction_v3.py'
url = 'https://raw.githubusercontent.com/Vedsaga/arc-agi/main/experiments/edit_schema_induction_v3.py'

r = requests.get(url, timeout=30)
r.raise_for_status()
module_path.write_bytes(r.content)
print('Downloaded:', url)

spec = importlib.util.spec_from_file_location('edit_schema_induction_v3', module_path)
v3 = importlib.util.module_from_spec(spec)
sys.modules['edit_schema_induction_v3'] = v3
spec.loader.exec_module(v3)
print('Loaded:', module_path)

## Run the frozen smoke test

The primary values to watch are:

- `demo_coverage_tasks`: at least one anti-unified schema exactly explains **all** demonstrations;
- `pass1_tasks` / `pass2_tasks`: exact held-out test solution;
- `anti_unified_schema_count`: residual ambiguity after explanation/generalization;
- runtime.

If coverage is still near zero, we should **not** keep patching this representation indefinitely. If coverage jumps materially above v0/v1, then the next ablation should compare backward induction against forward search using matched primitives and equal CPU budget.

In [ ]:
df, summary = v3.run(limit=LIMIT, split=SPLIT)
summary

In [ ]:
display(df.head(20))

print('\n=== RESULT ===')
print('tasks:', summary['tasks'])
print('demo coverage:', summary['demo_coverage_tasks'], '/', summary['tasks'])
print('pass@1:', summary['pass1_tasks'], '/', summary['tasks'])
print('pass@2:', summary['pass2_tasks'], '/', summary['tasks'])
print('median anti-unified schemas:', summary['median_anti_unified_schemas'])
print('median seconds/task:', summary['median_seconds'])
print('CSV:', summary['output_csv'])

## Files to send back

After the run, upload these two files from `/kaggle/working/edit_schema_v3/`:

- `edit_schema_v3_heldout.csv`
- `edit_schema_v3_heldout_summary.json`

Do **not** switch to the official evaluation set yet. If the 50-task smoke test shows real coverage, we will freeze the next ablation before scaling.